# pyQuil Tutorial: Noise and Quantum Computation

*Condensed from the [pyQuil docs, v4.17](https://pyquil-docs.rigetti.com/en/stable/noise.html).*

$\newcommand{\ket}[1]{\left|#1\right\rangle}\newcommand{\bra}[1]{\left\langle#1\right|}\newcommand{\tr}[1]{\mathrm{Tr}\left[#1\right]}$

> **Prerequisites:** `pip install pyquil`. Cells that run programs need the `qvm` and `quilc` servers running (the setup cell below starts them). Cells marked `# Requires QCS access` need a Rigetti Quantum Cloud Services account and will fail on a local-only setup.

Extra packages used here: `matplotlib`, `scipy`, `pandas`, `seaborn`.

## Setup

Use the **Python (Rigetti)** kernel (see the README for one-time kernel registration). Run the next cell once per session to start the `quilc` and `qvm` servers in the background (`!cmd &` doesn't work in Jupyter, so this uses `subprocess.Popen`).

In [ ]:
import subprocess, shutil, sys, pyquil

for cmd in ("quilc", "qvm"):
    exe = shutil.which(cmd) or f"/ocean/projects/pscstaff/nnepal/Rigetti/.venv/bin/{cmd}"
    subprocess.Popen([exe, "-S"], stdout=open(f"{cmd}.log", "w"), stderr=subprocess.STDOUT)

print(sys.executable, pyquil.__version__)

## Modeling noisy quantum gates

### Pure states vs. mixed states

Errors introduce *classical* uncertainty about the quantum state. When we don't know which state the system is in, we describe it as a **mixed state**: a probabilistic mixture of wavefunctions. Mixed states are represented by **density matrices**. A pure state's density matrix is

$$\rho_\psi = \ket{\psi}\bra{\psi}$$

and expectation values are

$$\langle X \rangle_\rho = \tr{X \rho}$$

Pure-state density matrices satisfy $\rho^2 = \rho$. Classical uncertainty between two states is a weighted sum,

$$\rho = p\,\rho_1 + (1-p)\,\rho_2$$

where $p \in [0,1]$ is the classical probability of $\rho_1$.

### Quantum gate errors

1. **Coherent errors** preserve purity: a perturbed unitary $\tilde{U} \neq U$ is applied instead of the ideal $U$.
2. **Incoherent errors** destroy purity. The state evolves as

$$\rho \mapsto \sum_{j=1}^m K_j\rho K_j^\dagger$$

where the **Kraus operators** $\{K_1, \dots, K_m\}$ satisfy $\sum_{j=1}^m K_j^\dagger K_j = I$ so that trace is conserved.

Coherent errors can in principle be fixed with better-calibrated control. Incoherent errors are trickier.

### Why do incoherent errors happen?

A qubit that isn't perfectly isolated co-evolves with its environment. The joint system-plus-environment evolution is unitary, but the system alone generally isn't. With $\mathcal{H} = \mathcal{H}_S \otimes \mathcal{H}_E$ and Hamiltonian

$$H = H_S \otimes I + I \otimes H_E + V$$

where $V$ couples system and environment, even a factorizable initial state evolves into an entangled one:

$$\ket{\psi_t} = e^{-i \frac{Ht}{\hbar}} \left(\ket{\psi}_{S,0}\otimes \ket{\psi}_{E,0}\right)$$

### A toy model

Assume $H_S = 0$, $H_E = 0$, weak coupling $\|tV/\hbar\|^2 \sim \epsilon \ll 1$, and $V = \sqrt{\epsilon}\, V_S \otimes V_E$ with $\bra{\psi}_{E,0} V_E\ket{\psi}_{E,0} = 0$. Joint evolution is

$$\rho \mapsto \rho' := e^{-i \frac{Vt}{\hbar}} \rho\, e^{+i \frac{Vt}{\hbar}}$$

Expanding to second order with Baker–Campbell–Hausdorff:

$$\rho' = \rho - \frac{it}{\hbar} [V, \rho] - \frac{t^2}{2\hbar^2} [V, [V, \rho]] + O(\epsilon^{3/2})$$

Tracing out the environment, with $\rho = \rho_{S,0} \otimes \rho_{E,0}$:

$$\rho_S' := \mathrm{Tr}_E[\rho'] = \rho_{S,0} - \frac{\gamma}{2} \left[ V_S^2\rho_{S,0} + \rho_{S,0} V_S^2 - 2 V_S\rho_{S,0}V_S\right]$$

where $\gamma := \frac{\epsilon t^2}{2\hbar^2}\tr{V_E^2 \rho_{E,0}} \ll 1$. This is approximately a Kraus map with $K_1 := I - \frac{\gamma}{2} V_S^2$ and $K_2 := \sqrt{\gamma}\, V_S$:

$$\rho_S \to \rho_S' = K_1\rho K_1^\dagger + K_2\rho K_2^\dagger = \rho - \frac{\gamma}{2}\left[ V_S^2 \rho + \rho V_S^2\right] + \gamma V_S\rho_S V_S + O(\gamma^2)$$

which agrees with the derivation to $O(\epsilon^{3/2})$. Similar arguments lead to the Lindblad master equation.

## Noisy gates on the QVM

You can annotate a Quil program with `PRAGMA`s that replace particular gates with imperfect (Kraus map) versions.

The QVM propagates *pure* states, so it simulates noisy gates correctly **on average over many runs**. When it applies a noisy gate, it randomly picks Kraus operator $K_j$ with probability

$$p_j := \bra{\psi}_S K_j^\dagger K_j \ket{\psi}_S$$

and produces

$$\ket{\psi'}_S = \frac{1}{\sqrt{p_j}} K_j \ket{\psi}_S$$

Over $N \gg 1$ runs, the $N_\ell \approx N p_\ell$ runs that picked $K_\ell$ average to

$$\overline{\rho_S'} \approx \sum_{\ell} K_{\ell}\ket{\psi}_S \bra{\psi}_S K_{\ell}^\dagger$$

So noisy simulations must be repeated many times to give representative results.

## Getting started

1. Develop a good noise model (e.g., from Quantum Process Tomography or Gate Set Tomography).
2. Define the Kraus operators as a list of numpy arrays: `kraus_ops = [K1, K2, ..., Km]`.
3. For a program `p`, call `p.define_noisy_gate("MY_NOISY_GATE", [q1, q2], kraus_ops)`, replacing the gate name and qubit indices.

In [ ]:
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np

from pyquil import Program, get_qc
from pyquil.gates import CZ, H, I, X, MEASURE
from pyquil.quilbase import Declare
from scipy.linalg import expm
from scipy.stats import binom

%matplotlib inline

In [ ]:
qc = get_qc('2q-qvm')

## Example 1: Amplitude damping

Amplitude damping is an imperfect identity map with Kraus operators

$$K_1 = \begin{pmatrix} 1 & 0 \\ 0 & \sqrt{1-p} \end{pmatrix}, \qquad K_2 = \begin{pmatrix} 0 & \sqrt{p} \\ 0 & 0 \end{pmatrix}$$

where $p$ is the probability that a qubit in $\ket{1}$ decays to $\ket{0}$.

In [ ]:
def damping_channel(damp_prob=.1):
    """
    Generate the Kraus operators corresponding to an amplitude damping
    noise channel.

    :params float damp_prob: The one-step damping probability.
    :return: A list [k1, k2] of the Kraus operators that parametrize the map.
    :rtype: list
    """
    damping_op = np.sqrt(damp_prob) * np.array([[0, 1],
                                                [0, 0]])

    residual_kraus = np.diag([1, np.sqrt(1-damp_prob)])
    return [residual_kraus, damping_op]

def append_kraus_to_gate(kraus_ops, g):
    """
    Follow a gate `g` by a Kraus map described by `kraus_ops`.

    :param list kraus_ops: The Kraus operators.
    :param numpy.ndarray g: The unitary gate.
    :return: A list of transformed Kraus operators.
    """
    return [kj.dot(g) for kj in kraus_ops]

def append_damping_to_gate(gate, damp_prob=.1):
    """
    Generate the Kraus operators corresponding to a given unitary
    single qubit gate followed by an amplitude damping noise channel.

    :params np.ndarray|list gate: The 2x2 unitary gate matrix.
    :params float damp_prob: The one-step damping probability.
    :return: A list [k1, k2] of the Kraus operators that parametrize the map.
    :rtype: list
    """
    return append_kraus_to_gate(damping_channel(damp_prob), gate)

In [ ]:
%%time
# single step damping probability
damping_per_I = 0.02

# number of program executions
trials = 200

results_damping = []
lengths = np.arange(0, 201, 10, dtype=int)
for jj, num_I in enumerate(lengths):

    p = Program(
        Declare("ro", "BIT", 1),
        X(0),
    )
    # want increasing number of I-gates
    p.inst([I(0) for _ in range(num_I)])
    p.inst(MEASURE(0, ("ro", 0)))

    # overload identity I on qc 0
    p.define_noisy_gate("I", [0], append_damping_to_gate(np.eye(2), damping_per_I))
    p.wrap_in_numshots_loop(trials)
    qc.qam.random_seed = int(num_I)
    res = qc.run(p).get_register_map().get("ro")
    results_damping.append([np.mean(res), np.std(res) / np.sqrt(trials)])

results_damping = np.array(results_damping)

In [ ]:
dense_lengths = np.arange(0, lengths.max()+1, .2)
survival_probs = (1-damping_per_I)**dense_lengths
logpmf = binom.logpmf(np.arange(trials+1)[np.newaxis, :], trials,
                      survival_probs[:, np.newaxis])/np.log(10)

In [ ]:
DARK_TEAL = '#48737F'
FUSCHIA = "#D6619E"
BEIGE = '#EAE8C6'
cm = colors.LinearSegmentedColormap.from_list('anglemap', ["white", FUSCHIA, BEIGE], N=256, gamma=1.5)

plt.figure(figsize=(14, 6))
plt.pcolor(dense_lengths, np.arange(trials+1)/trials, logpmf.T, cmap=cm, vmin=-4, vmax=logpmf.max())
plt.plot(dense_lengths, survival_probs, c=BEIGE, label="Expected mean")
plt.errorbar(lengths, results_damping[:,0], yerr=2*results_damping[:,1], c=DARK_TEAL,
             label=r"noisy qvm, errorbars $ = \pm 2\hat{\sigma}$", marker="o")
cb = plt.colorbar()
cb.set_label(r"$\log_{10} \mathrm{Pr}(n_1; n_{\rm trials}, p_{\rm survival}(t))$", size=20)

plt.title("Amplitude damping model of a single qubit", size=20)
plt.xlabel(r"Time $t$ [arb. units]", size=14)
plt.ylabel(r"$n_1/n_{\rm trials}$", size=14)
plt.legend(loc="best", fontsize=18)
plt.xlim(*lengths[[0, -1]])
plt.ylim(0, 1)

## Example 2: Dephased CZ gate

Dephasing is characterized by a qubit's $T_2$ time. Single-qubit dephasing has Kraus operators

$$K_1(p) = \sqrt{1-p}\, I_2, \qquad K_2(p) = \sqrt{p}\, \sigma_Z$$

where $p = (1 - \exp(-T_{\rm gate}/T_2))/2$ is the dephasing probability over one gate interval.

For two qubits there are four outcomes — no dephasing, qubit 1 dephases, qubit 2 dephases, both dephase:

$$K'_1 = K_1(p)\otimes K_1(q),\quad K'_2 = K_2(p)\otimes K_1(q),\quad K'_3 = K_1(p)\otimes K_2(q),\quad K'_4 = K_2(p)\otimes K_2(q)$$

Both dephasing and CZ are diagonal, so we compose $U_{\rm CZ}$ with each operator: $K^{\rm CZ}_j = K'_j\, U_{\rm CZ}$.

> **Note:** This isn't always accurate — real CZ gates often use non-diagonal interaction Hamiltonians — but it's a good starting point for small dephasing probabilities.

In [ ]:
def dephasing_kraus_map(p=.1):
    """
    Generate the Kraus operators corresponding to a dephasing channel.

    :params float p: The one-step dephasing probability.
    :return: A list [k1, k2] of the Kraus operators that parametrize the map.
    :rtype: list
    """
    return [np.sqrt(1-p)*np.eye(2), np.sqrt(p)*np.diag([1, -1])]

def tensor_kraus_maps(k1, k2):
    """
    Generate the Kraus map corresponding to the composition
    of two maps on different qubits.

    :param list k1: The Kraus operators for the first qubit.
    :param list k2: The Kraus operators for the second qubit.
    :return: A list of tensored Kraus operators.
    """
    return [np.kron(k1j, k2l) for k1j in k1 for k2l in k2]

In [ ]:
%%time
# single step damping probabilities
ps = np.linspace(.001, .5, 200)

# number of program executions
trials = 500

results = []

for jj, p in enumerate(ps):

    corrupted_CZ = append_kraus_to_gate(
        tensor_kraus_maps(
            dephasing_kraus_map(p),
            dephasing_kraus_map(p)
        ),
        np.diag([1, 1, 1, -1]))

    # make Bell-state
    p = Program(
        Declare("ro", "BIT", 2),
        H(0),
        H(1),
        CZ(0, 1),
        H(1),
    )

    p.inst(MEASURE(0, ("ro", 0)))
    p.inst(MEASURE(1, ("ro", 1)))

    # overload CZ on qc 0
    p.define_noisy_gate("CZ", [0, 1], corrupted_CZ)
    p.wrap_in_numshots_loop(trials)
    qc.qam.random_seed = jj
    res = qc.run(p).get_register_map().get("ro")
    results.append(res)

results = np.array(results)

In [ ]:
Z1s = (2*results[:,:,0]-1.)
Z2s = (2*results[:,:,1]-1.)
Z1Z2s = Z1s * Z2s

Z1m = np.mean(Z1s, axis=1)
Z2m = np.mean(Z2s, axis=1)
Z1Z2m = np.mean(Z1Z2s, axis=1)

In [ ]:
plt.figure(figsize=(14, 6))
plt.axhline(y=1.0, color=FUSCHIA, alpha=.5, label="Bell state")

plt.plot(ps, Z1Z2m, "x", c=FUSCHIA, label=r"$\overline{Z_1 Z_2}$")
plt.plot(ps, 1-2*ps, "--", c=FUSCHIA, label=r"$\langle Z_1 Z_2\rangle_{\rm theory}$")

plt.plot(ps, Z1m, "o", c=DARK_TEAL, label=r"$\overline{Z}_1$")
plt.plot(ps, 0*ps, "--", c=DARK_TEAL, label=r"$\langle Z_1\rangle_{\rm theory}$")

plt.plot(ps, Z2m, "d", c="k", label=r"$\overline{Z}_2$")
plt.plot(ps, 0*ps, "--", c="k", label=r"$\langle Z_2\rangle_{\rm theory}$")

plt.xlabel(r"Dephasing probability $p$", size=18)
plt.ylabel(r"$Z$-moment", size=18)
plt.title(r"$Z$-moments for a Bell-state prepared with dephased CZ", size=18)
plt.xlim(0, .5)
plt.legend(fontsize=18)

## Adding decoherence noise

Here we look at how a program behaves on a near-term device with $T_1$- and $T_2$-type noise, using `pyquil.noise.add_decoherence_noise()`. The `pyquil.noise` module also has `tensor_kraus_maps()` (multi-qubit noise), `combine_kraus_maps()` (successive noise processes) and `append_kraus_to_gate()` (appending noise to a unitary gate).

In [ ]:
from pyquil.quil import Program
from pyquil.paulis import PauliSum, PauliTerm, exponentiate, exponential_map, trotterize
from pyquil.gates import MEASURE, H, Z, RX, RZ, CZ
from pyquil.quilbase import Declare
import numpy as np

### The task

Prepare $e^{i \theta XY}$ and measure it in the $Z$ basis.

In [ ]:
from numpy import pi
theta = pi/3
xy = PauliTerm('X', 0) * PauliTerm('Y', 1)

### The idiomatic pyQuil program

In [ ]:
prog = exponential_map(xy)(theta)
print(prog)

### The compiled program

Real devices (and this noise model) use native gates: $I$, $RZ(\theta)$, $RX(\pm \pi/2)$, $CZ$. Here is the compiled version:

In [ ]:
def get_compiled_prog(theta):
    return Program([
        RZ(-pi/2, 0),
        RX(-pi/2, 0),
        RZ(-pi/2, 1),
        RX( pi/2, 1),
        CZ(1, 0),
        RZ(-pi/2, 1),
        RX(-pi/2, 1),
        RZ(theta, 1),
        RX( pi/2, 1),
        CZ(1, 0),
        RX( pi/2, 0),
        RZ( pi/2, 0),
        RZ(-pi/2, 1),
        RX( pi/2, 1),
        RZ(-pi/2, 1),
    ])

### Scan over noise parameters

We scan three $T_1$ values (1, ≈3.16 and 10 µs) at 20 values of θ. By default $T_2 = T_1/2$, 1-qubit gates take 50 ns and 2-qubit gates take 150 ns. Identity and parametric $RZ$ gates are noiseless; $RX$ and $CZ$ get 1-qubit and 2-qubit noise respectively.

In [ ]:
from pyquil import get_qc
qc = get_qc("2q-qvm")

In [ ]:
t1s = np.logspace(-6, -5, num=3)
thetas = np.linspace(-pi, pi, num=20)
t1s * 1e6  # us

In [ ]:
from pyquil.noise import add_decoherence_noise
records = []
for theta in thetas:
    for t1 in t1s:
        prog = get_compiled_prog(theta)
        noisy = add_decoherence_noise(prog, T1=t1, T2=t1/2).inst([
            Declare("ro", "BIT", 2),
            MEASURE(0, ("ro", 0)),
            MEASURE(1, ("ro", 1)),
        ])
        bitstrings = qc.run(noisy).get_register_map().get("ro")

        # Expectation of Z0 and Z1
        z0, z1 = 1 - 2*np.mean(bitstrings, axis=0)

        # Expectation of ZZ by computing the parity of each pair
        zz = 1 - (np.sum(bitstrings, axis=1) % 2).mean() * 2

        record = {
            'z0': z0,
            'z1': z1,
            'zz': zz,
            'theta': theta,
            't1': t1,
        }
        records += [record]

### Plot the results (requires `pandas` and `seaborn`)

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(style='ticks', palette='colorblind')

In [ ]:
import pandas as pd
df_all = pd.DataFrame(records)
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12,4))

for t1 in t1s:
    df = df_all.query('t1 == @t1')

    ax1.plot(df['theta'], df['z0'], 'o-')
    ax2.plot(df['theta'], df['z1'], 'o-')
    ax3.plot(df['theta'], df['zz'], 'o-', label='T1 = {:.0f} us'.format(t1*1e6))

ax3.legend(loc='best')

ax1.set_ylabel('Z0')
ax2.set_ylabel('Z1')
ax3.set_ylabel('ZZ')
ax2.set_xlabel(r'$\theta$')
fig.tight_layout()

## Modeling readout noise

Readout is corrupted by two main mechanisms:

1. **Classical readout bit-flip error** — noise on the transmission line makes a 0 look like a 1 or vice versa. Reduced by optimized readout pulses and quantum-limited amplifiers.
2. **T1-readout error** — the qubit decays during readout (which can exceed a microsecond), so a signal that starts like $\ket{1}$ ends up like $\ket{0}$. Reduced by shortening readout relative to $T_1$.

### Projective, ideal measurement

Born's rule and the projection postulate give a complete set of orthogonal projectors $P_{\mathcal{O}} := \{\Pi_x \mid x \in \mathcal{O}\}$ with $\Pi_x^\dagger = \Pi_x = \Pi_x^2$, $\Pi_x\Pi_y = \delta_{xy}\Pi_x$ and $\sum_x \Pi_x = 1$. The probability of outcome $x$ is

$$p(x) = \tr{\Pi_x \rho \Pi_x} = \tr{\Pi_x \rho}$$

and the post-measurement state is $\rho_x = \frac{1}{p(x)} \Pi_x \rho \Pi_x$. If the outcome is discarded, the best guess of the post-measurement state is

$$\rho_{\text{post measurement}} = \sum_{x\in\mathcal{O}} p(x) \rho_x = \sum_{x\in\mathcal{O}} \Pi_x \rho \Pi_x$$

### Classical readout bit-flip error

An ideal outcome $x$ passes through a noisy classical channel and becomes $x' \in \{0', 1'\}$, described by the **assignment probability matrix**

$$P_{x'|x} = \begin{pmatrix} p(0 | 0) & p(0 | 1) \\ p(1 | 0) & p(1 | 1) \end{pmatrix}$$

which has two free parameters (each column is a probability distribution). Probabilities transform as $\mathbf{p}' = P_{x'|x}\mathbf{p}$. Using Bayes' theorem, the noisy outcome probabilities are $p(x') = \tr{\rho E_{x'}}$ with

$$E_{x'} := \sum_{x\in \mathcal{O}} p(x'|x)\Pi_x \ge 0, \qquad \sum_{x'} E_{x'} = 1$$

The family $\{E_{x'}\}$ is a **positive operator valued measure (POVM)**. Its elements generally aren't orthogonal projectors, but they arise naturally here.

### How to model $T_1$ error

$T_1$ error corrupts the state during measurement and can erase pre-measurement information. The most general description uses measurement instruments; in practice, model it as a $T_1$ damping Kraus map followed by the noisy readout above.

*Further reading:* Chapter 3 of [John Preskill's lecture notes](http://theory.caltech.edu/~preskill/ph219/chap3_15.pdf).

## Working with readout noise

1. Estimate $p(0|0)$ and $p(1|1)$ (the off-diagonals follow from normalization). Given an assignment fidelity $F$ and roughly symmetric bit-flip noise, set $p(0|0)=p(1|1)=F$.
2. For program `p` and qubit `q`, call `p.define_noisy_readout(q, p00, p11)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyquil import get_qc
from pyquil.quil import Program, MEASURE, Pragma
from pyquil.gates import I, X, RX, H, CNOT
from pyquil.noise import (estimate_bitstring_probs, correct_bitstring_probs,
                          bitstring_probs_to_z_moments, estimate_assignment_probs)

DARK_TEAL = '#48737F'
FUSCHIA = '#D6619E'
BEIGE = '#EAE8C6'

qc = get_qc("1q-qvm")

### Example 1: Rabi sequence with noisy readout

In [ ]:
%%time
# number of angles
num_theta = 101

# number of program executions
trials = 200

thetas = np.linspace(0, 2*np.pi, num_theta)

p00s = [1., 0.95, 0.9, 0.8]

results_rabi = np.zeros((num_theta, len(p00s)))

for jj, theta in enumerate(thetas):
    for kk, p00 in enumerate(p00s):
        qc.qam.random_seed = 1
        p = Program(RX(theta, 0)).wrap_in_numshots_loop(trials)
        # assume symmetric noise p11 = p00
        p.define_noisy_readout(0, p00=p00, p11=p00)
        ro = p.declare("ro", "BIT", 1)
        p.measure(0, ro[0])
        res = qc.run(p).get_register_map().get("ro")
        results_rabi[jj, kk] = np.sum(res)

In [ ]:
plt.figure(figsize=(14, 6))
for jj, (p00, c) in enumerate(zip(p00s, [DARK_TEAL, FUSCHIA, "k", "gray"])):
    plt.plot(thetas, results_rabi[:, jj]/trials, c=c, label=r"$p(0|0)=p(1|1)={:g}$".format(p00))
plt.legend(loc="best")
plt.xlim(*thetas[[0,-1]])
plt.ylim(-.1, 1.1)
plt.grid(alpha=.5)
plt.xlabel(r"RX angle $\theta$ [radian]", size=16)
plt.ylabel(r"Excited state fraction $n_1/n_{\rm trials}$", size=16)
plt.title("Effect of classical readout noise on Rabi contrast.", size=18)

### Example 2: Estimate the assignment probabilities

We estimate $P_{x'|x}$ experimentally. On a perfect quantum computer we expect the identity matrix:

In [ ]:
print(estimate_assignment_probs(0, 1000, qc))

On an imperfect quantum computer (simulated with noisy-readout headers). The docs use qubits 0–2 here, so we switch to a 3-qubit QVM:

In [ ]:
qc = get_qc("3q-qvm")
qc.qam.random_seed = None
header0 = Program().define_noisy_readout(0, .85, .95)
header1 = Program().define_noisy_readout(1, .8, .9)
header2 = Program().define_noisy_readout(2, .9, .85)

ap0 = estimate_assignment_probs(0, 10000, qc, header0)
ap1 = estimate_assignment_probs(1, 10000, qc, header1)
ap2 = estimate_assignment_probs(2, 10000, qc, header2)

In [ ]:
print(ap0, ap1, ap2, sep="\n")

### Example 3: Correct for noisy readout

#### Correcting the Rabi signal from above

In [ ]:
ap_last = np.array([[p00s[-1], 1 - p00s[-1]],
                    [1 - p00s[-1], p00s[-1]]])
corrected_last_result = [correct_bitstring_probs([1-p, p], [ap_last])[1] for p in results_rabi[:, -1] / trials]

In [ ]:
plt.figure(figsize=(14, 6))
for jj, (p00, c) in enumerate(zip(p00s, [DARK_TEAL, FUSCHIA, "k", "gray"])):
    if jj not in [0, 3]:
        continue
    plt.plot(thetas, results_rabi[:, jj]/trials, c=c, label=r"$p(0|0)=p(1|1)={:g}$".format(p00), alpha=.3)
plt.plot(thetas, corrected_last_result, c="red", label=r"Corrected $p(0|0)=p(1|1)={:g}$".format(p00s[-1]))
plt.legend(loc="best")
plt.xlim(*thetas[[0,-1]])
plt.ylim(-.1, 1.1)
plt.grid(alpha=.5)
plt.xlabel(r"RX angle $\theta$ [radian]", size=16)
plt.ylabel(r"Excited state fraction $n_1/n_{\rm trials}$", size=16)
plt.title("Corrected contrast", size=18)

The corrected signal is fairly noisy (and sometimes leaves $[0,1]$) because of the small sample size, $n=200$.

#### Corrupting and correcting GHZ state correlations

We prepare the GHZ state $\frac{1}{\sqrt{2}}\left[\ket{000} + \ket{111}\right]$, measure outcome probabilities with and without readout noise, and see how Pauli-Z moments (qubit correlations) are corrupted and then corrected.

In [ ]:
from pyquil.quilbase import Declare
from pyquil.gates import MEASURE

ghz_prog = Program(
    Declare("ro", "BIT", 3),
    H(0),
    CNOT(0, 1),
    CNOT(1, 2),
    MEASURE(0, ("ro", 0)),
    MEASURE(1, ("ro", 1)),
    MEASURE(2, ("ro", 2)),
)
ghz_prog.wrap_in_numshots_loop(10000)
print(ghz_prog)
results = qc.run(ghz_prog).get_register_map().get("ro")

In [ ]:
header = header0 + header1 + header2
noisy_ghz = header + ghz_prog
noisy_ghz.wrap_in_numshots_loop(10000)
print(noisy_ghz)
noisy_results = qc.run(noisy_ghz).get_register_map().get("ro")

**Uncorrupted** probability of $\ket{000}$ and $\ket{111}$ — each should be ≈ 1/2:

In [ ]:
probs = estimate_bitstring_probs(results)
probs[0, 0, 0], probs[1, 1, 1]

**Corrupted** probabilities deviate significantly from the ideal values:

In [ ]:
noisy_probs = estimate_bitstring_probs(noisy_results)
noisy_probs[0, 0, 0], noisy_probs[1, 1, 1]

**Corrected** probabilities are much closer to ideal:

In [ ]:
corrected_probs = correct_bitstring_probs(noisy_probs, [ap0, ap1, ap2])
corrected_probs[0, 0, 0], corrected_probs[1, 1, 1]

Estimate $\langle Z_0^{j} Z_1^{k} Z_2^{\ell}\rangle$ for $jk\ell = 100, 010, 001$ from **non-noisy** data — we expect these all to be very small:

In [ ]:
zmoments = bitstring_probs_to_z_moments(probs)
zmoments[1, 0, 0], zmoments[0, 1, 0], zmoments[0, 0, 1]

…and for $jk\ell = 110, 011, 101$ — we expect these all to be close to 1:

In [ ]:
zmoments[1, 1, 0], zmoments[0, 1, 1], zmoments[1, 0, 1]

The same moments from **noise-corrected** data:

In [ ]:
zmoments_corr = bitstring_probs_to_z_moments(corrected_probs)
zmoments_corr[1, 0, 0], zmoments_corr[0, 1, 0], zmoments_corr[0, 0, 1]

In [ ]:
zmoments_corr[1, 1, 0], zmoments_corr[0, 1, 1], zmoments_corr[1, 0, 1]

Correction restores the contrast of multi-qubit observables, though it can give slightly non-physical values; more samples reduce this.

## Alternative: a global Pauli error model

The QVM can also emulate **parametric Pauli noise**, defined by six probabilities:

- $P_X$, $P_Y$, $P_Z$ — probability of a Pauli $X$, $Y$ or $Z$ being applied to each qubit after every gate (gate noise)
- $P_X'$, $P_Y'$, $P_Z'$ — probability of a Pauli $X$, $Y$ or $Z$ being applied to a qubit right before it's measured (measurement noise)

In [ ]:
noisy_qc = get_qc("1q-qvm")
noisy_qc.qam.gate_noise = (0.2, 0.0, 0.0)
noisy_qc.qam.measurement_noise = (0.2, 0.0, 0.0)

Apply an `X` gate and measure. Without noise we should always get `1`:

In [ ]:
p = Program(
    Declare("ro", "BIT", 1),
    X(0),
    MEASURE(0, ("ro", 0)),
).wrap_in_numshots_loop(10)

qc = get_qc("1q-qvm")
print("Without Noise:")
print(qc.run(p).get_register_map().get("ro"))
print("With Noise:")
print(noisy_qc.run(p).get_register_map().get("ro"))